# Move your February notebook to your computer

*Python Foundations · Topic 10.1*

The marketing team's dashboard shows sessions by source / medium, and they want the same top five for February 2024 from the raw export, to check the dashboard against. The two cells after the first one are from a Colab notebook that loaded the export from Google Drive. Make them work on your computer, then answer the question.

This workbook is a file in your project, so JupyterLab saves your changes in it. Save with Cmd+S (Ctrl+S on Windows).

1. Run the first code cell. It shows the Python that runs this notebook, which should be in your `referral-report` project's `.venv`.
2. Delete the cell that mounts Google Drive: click it, and choose **Edit → Delete Cell**.
3. In the cell that reads the file, change `CSV_PATH` to the file's path relative to this notebook. `get_files.py` put the file in the `data` folder. Run the cell: `len(rows)` should be `1724`.
4. In the first cell marked ✏️, create `sessions_by_source_medium`: a dictionary whose keys are `(session_source, session_medium)` tuples, with each pair's total sessions.
5. In the second cell marked ✏️, create `top_five`, the five pairs with the most sessions, with `Counter`. Then print each one as `source / medium: sessions`.
6. Run the **Check your work** cell.
7. Choose **Kernel → Restart Kernel and Run All Cells…**, and check that you get the same result. Save the notebook, and shut JupyterLab down with **File → Shut Down**.

**Expected output**: `sessions_by_source_medium` has 24 keys, and the last cell prints:

    google / organic: 18199
    google / cpc: 12611
    (direct) / (none): 12313
    youtube / video: 9856
    facebook / paid_social: 6874

In [ ]:
import sys

print(sys.executable)

In [ ]:
import csv

CSV_PATH = "data/sessions_export_2024_02.csv"

with open(CSV_PATH, newline="", encoding="utf-8") as file:
    rows = list(csv.DictReader(file))
len(rows)

In [ ]:
sessions_by_source_medium = {}
for row in rows:
    key = (row["session_source"], row["session_medium"])
    sessions_by_source_medium[key] = sessions_by_source_medium.get(key, 0) + int(row["sessions"])
len(sessions_by_source_medium)

In [ ]:
from collections import Counter

top_five = Counter(sessions_by_source_medium).most_common(5)
for (source, medium), sessions in top_five:
    print(f"{source} / {medium}: {sessions}")

In [ ]:
# Check your work: run this cell last. If you change anything above, run the cells again from the top first.
# This cell only reads your variables. It doesn't change anything.
def check_your_work():
    import os
    import sys
    from pathlib import Path
    found = globals()
    results = []

    folder = Path.cwd()
    project = next((path for path in [folder, *folder.parents] if (path / "pyproject.toml").exists()), None)
    prefix = Path(sys.prefix).resolve()
    if project is None:
        results.append((False, f"This notebook isn't in a uv project's folder, but in {folder}. Move it into referral-report, with its data folder, and start JupyterLab there with uv run jupyter lab."))
    elif project.name != "referral-report":
        results.append((False, f"This notebook is in the {project.name} project. The exercise uses referral-report, from topic 9.3.: run get_files.py in that folder, and start JupyterLab there."))
    elif prefix == (project / ".venv").resolve():
        results.append((True, "The kernel is referral-report's .venv."))
    elif "archive-v0" in prefix.parts or prefix.parent.name == "tools":
        results.append((False, f"The kernel is {sys.executable}, the Python of a JupyterLab started with uvx or installed as a uv tool, not the project's. Shut JupyterLab down and start it in referral-report with uv run jupyter lab."))
    else:
        results.append((False, f"The kernel is {sys.executable}, not referral-report's .venv. Shut JupyterLab down and start it in referral-report with uv run jupyter lab."))

    if "google.colab" in sys.modules:
        results.append((False, "google.colab is imported, so this ran in Colab. This exercise is for JupyterLab on your computer."))

    path = found.get("CSV_PATH")
    if not isinstance(path, str):
        results.append((False, "CSV_PATH isn't defined. Run the cell that reads the file."))
    elif path.startswith("/content"):
        results.append((False, f"CSV_PATH is still Colab's path, {path!r}. The file is in the data folder next to this notebook: \"data/sessions_export_2024_02.csv\"."))
    elif Path(path).is_absolute():
        results.append((False, f"CSV_PATH is {path!r}, which only works on your computer. Use the path relative to this notebook: \"data/sessions_export_2024_02.csv\"."))
    elif not path.replace("\\", "/").endswith("sessions_export_2024_02.csv"):
        results.append((False, f"CSV_PATH is {path!r}. The exercise uses February's export, sessions_export_2024_02.csv."))
    elif not os.path.exists(path):
        results.append((False, f"There's no file at {path!r}, relative to {folder}. Check that the data folder is next to this notebook. If it isn't there, run get_files.py again."))
    else:
        results.append((True, "CSV_PATH is a relative path to February's export."))

    rows = found.get("rows")
    if not isinstance(rows, list):
        results.append((False, "rows isn't a list. Run the cell that reads the file."))
    elif len(rows) == 1724:
        results.append((True, "rows has all 1,724 rows of the export."))
    else:
        results.append((False, f"rows has {len(rows)} rows, but sessions_export_2024_02.csv has 1,724. Read February's file with csv.DictReader."))

    totals = found.get("sessions_by_source_medium")
    if totals is None:
        results.append((False, "sessions_by_source_medium isn't defined. Run the cell that creates it."))
    elif not isinstance(totals, dict):
        results.append((False, f"sessions_by_source_medium should be a dictionary, but it is a {type(totals).__name__}."))
    elif not totals:
        results.append((False, "sessions_by_source_medium is empty. If its cell stopped with an error, read the error, fix it and run the cell again."))
    elif not all(isinstance(key, tuple) for key in totals):
        key = next(key for key in totals if not isinstance(key, tuple))
        results.append((False, f"sessions_by_source_medium has the key {key!r}. Use a tuple of the two values as the key: (row[\"session_source\"], row[\"session_medium\"])."))
    elif ("cpc", "google") in totals:
        results.append((False, "The keys are (medium, source). Put the source first: (row[\"session_source\"], row[\"session_medium\"])."))
    elif len(totals) != 24:
        results.append((False, f"sessions_by_source_medium has {len(totals)} keys, but February has 24 source / medium pairs. Use the session_source and session_medium of every row."))
    elif sum(value for value in totals.values() if isinstance(value, int)) == 1724:
        results.append((False, "sessions_by_source_medium counts rows, not sessions. Add int(row[\"sessions\"]) to the pair's total."))
    elif totals.get(("google", "cpc")) == 12611 and sum(totals.values()) == 84654:
        results.append((True, "sessions_by_source_medium has the right sessions for all 24 pairs."))
    else:
        results.append((False, f"sessions_by_source_medium[('google', 'cpc')] should be 12611, but it is {totals.get(('google', 'cpc'))!r}. Add each row's sessions to its pair's total with .get(key, 0) + int(row[\"sessions\"]), instead of replacing the total."))

    top_five = found.get("top_five")
    expected = [(("google", "organic"), 18199), (("google", "cpc"), 12611), (("(direct)", "(none)"), 12313), (("youtube", "video"), 9856), (("facebook", "paid_social"), 6874)]
    if top_five is None:
        results.append((False, "top_five isn't defined. Run the cell that creates it."))
    elif top_five == expected:
        results.append((True, "top_five has the five pairs with the most sessions."))
    elif isinstance(top_five, dict):
        results.append((False, "top_five is a dictionary. Use Counter(sessions_by_source_medium).most_common(5), which gives a list of (key, sessions) tuples."))
    elif isinstance(top_five, list) and len(top_five) != 5:
        results.append((False, f"top_five has {len(top_five)} pairs. Ask most_common() for 5."))
    elif isinstance(top_five, list) and top_five == expected[::-1]:
        results.append((False, "top_five is in reverse order. most_common() puts the biggest first, so use its list as it is."))
    else:
        results.append((False, f"top_five should be {expected}, but it is {top_five!r}."))

    for passed, message in results:
        print(("✅ " if passed else "❌ ") + message)
    if all(passed for passed, message in results):
        print("\nEverything matches. Now choose Kernel → Restart Kernel and Run All Cells…, and check that you get the same result.")
    else:
        print("\nFix the first ❌, run the cells above again from the top, then run this cell again.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

Delete the cell that mounts Google Drive.

```python
import csv

CSV_PATH = "data/sessions_export_2024_02.csv"

with open(CSV_PATH, newline="", encoding="utf-8") as file:
    rows = list(csv.DictReader(file))
len(rows)
```

```python
sessions_by_source_medium = {}
for row in rows:
    key = (row["session_source"], row["session_medium"])
    sessions_by_source_medium[key] = sessions_by_source_medium.get(key, 0) + int(row["sessions"])
len(sessions_by_source_medium)
```

```python
from collections import Counter

top_five = Counter(sessions_by_source_medium).most_common(5)
for (source, medium), sessions in top_five:
    print(f"{source} / {medium}: {sessions}")
```

```text
cd ~/simmer-python/referral-report
uv add --dev jupyterlab
uv run https://raw.githubusercontent.com/team-simmer/python-foundations/main/lesson-10/get_files.py 10.1
uv run jupyter lab
```

</details>